# BIOT 6900 · Module 2 · Part 2 — Load Real CPTAC Breast Cancer Data
### Run this notebook to turn the LinkedOmics downloads into ready-to-use files. **You don't write any code.**

**Do this:**
1. Download the three files from **https://www.linkedomics.org/data_download/CPTAC-BRCA/**
   - `HS_CPTAC_BRCA_2018_RNA_GENE.cct`
   - `HS_CPTAC_BRCA_2018_Proteome_Ratio_Norm_gene_Median.cct`
   - `HS_CPTAC_BRCA_2018_MUT_GENE.cbt`
2. Put all three in **`data/part2_cptac_brca_raw/`**.
3. `Kernel → Restart & Run All`.

That's it. This notebook orients each file correctly, collapses the binary mutation matrix into a per-gene frequency, and writes the three files the main notebook expects into `data/part1_cptac_brca/`:
`cptac_brca_rna.tsv`, `cptac_brca_protein.tsv`, `cptac_brca_mutation.tsv`.

*(Prefer not to download by hand? Set `AUTO_DOWNLOAD = True` in the config cell and it will fetch the files from LinkedOmics for you.)*

In [6]:
# ===== CONFIG — you normally change nothing here =====
# Put the three files you downloaded from LinkedOmics in data/part2_cptac_brca_raw/.
# Then just Run All. That's it.
RAW_DIR = "data/part2_cptac_brca_raw"  # folder holding the three downloaded files
OUT_DIR = "data/part1_cptac_brca"      # where the ready-to-use files are written (Part 1 reads from here)
AUTO_DOWNLOAD = False  # set to True to fetch straight from LinkedOmics instead of downloading by hand

FILES = {
    "rna":  "HS_CPTAC_BRCA_2018_RNA_GENE.cct",
    "prot": "HS_CPTAC_BRCA_2018_Proteome_Ratio_Norm_gene_Median.cct",
    "mut":  "HS_CPTAC_BRCA_2018_MUT_GENE.cbt",
}
BASE_URL = "https://www.linkedomics.org/data_download/CPTAC-BRCA/"

In [11]:
import os
import pandas as pd

os.makedirs(OUT_DIR, exist_ok=True)

def read_matrix(name):
    """Read a LinkedOmics .cct/.cbt matrix from disk (or LinkedOmics) and
    orient it so GENES are the rows. No manual steps needed."""
    src = (BASE_URL + name) if AUTO_DOWNLOAD else os.path.join(RAW_DIR, name)
    try:
        df = pd.read_csv(src, sep="\t", index_col=0)
    except Exception as e:
        where = "LinkedOmics" if AUTO_DOWNLOAD else f"the folder '{RAW_DIR}'"
        raise FileNotFoundError(
            f"Could not read {name} from {where}.\n"
            f"  - If downloading by hand: put the file in {RAW_DIR}/.\n"
            f"  - If AUTO_DOWNLOAD=True: check your internet connection.\n"
            f"Details: {e}")
    # genes are the long axis (thousands) vs ~122 samples; make genes the rows
    if df.shape[0] < df.shape[1]:
        df = df.T
    return df

In [12]:
# --- RNA and protein: just orient + rename ---
rna = read_matrix(FILES["rna"])
rna.to_csv(os.path.join(OUT_DIR, "cptac_brca_rna.tsv"), sep="\t")
print("RNA     ->", rna.shape, "(genes x samples)")

prot = read_matrix(FILES["prot"])
prot.to_csv(os.path.join(OUT_DIR, "cptac_brca_protein.tsv"), sep="\t")
print("Protein ->", prot.shape, "(genes x samples)")

# --- Mutation: binary matrix -> one number per gene (fraction of tumors mutated) ---
mut = read_matrix(FILES["mut"]).apply(pd.to_numeric, errors="coerce")
mut_freq = mut.mean(axis=1)
mut_freq.name = "mut_freq"
mut_freq.to_frame().to_csv(os.path.join(OUT_DIR, "cptac_brca_mutation.tsv"), sep="\t")
print("Mutation->", mut_freq.shape[0], "genes (collapsed to per-gene frequency)")

RNA     -> (23121, 122) (genes x samples)
Protein -> (12621, 122) (genes x samples)
Mutation-> 9448 genes (collapsed to per-gene frequency)


In [13]:
import os
print("RAW_DIR =", RAW_DIR)
print(sorted(os.listdir(RAW_DIR)))
print("mut filename code expects:", FILES["mut"])

RAW_DIR = .
['.DS_Store', '.git', 'BIOT6900_Module2_Part2_LoadRealCPTAC copy.ipynb', 'BIOT6900_Module2_Part2_LoadRealCPTAC.ipynb', 'BIOT6900_Module2_Starter.ipynb', 'HS_CPTAC_BRCA_2018_MUT_GENE.cbt', 'HS_CPTAC_BRCA_2018_Proteome_Ratio_Norm_gene_Median.cct', 'HS_CPTAC_BRCA_2018_RNA_GENE.cct', 'README.md', 'data', 'module1_setup.ipynb']
mut filename code expects: HS_CPTAC_BRCA_2018_MUT_GENE.cbt


In [14]:
import os
for f in os.listdir(RAW_DIR):
    if f.endswith(".cct.txt") or f.endswith(".cbt.txt"):
        os.rename(os.path.join(RAW_DIR, f),
                  os.path.join(RAW_DIR, f[:-4]))   # drop the ".txt"
print(sorted(os.listdir(RAW_DIR)))

['.DS_Store', '.git', 'BIOT6900_Module2_Part2_LoadRealCPTAC copy.ipynb', 'BIOT6900_Module2_Part2_LoadRealCPTAC.ipynb', 'BIOT6900_Module2_Starter.ipynb', 'HS_CPTAC_BRCA_2018_MUT_GENE.cbt', 'HS_CPTAC_BRCA_2018_Proteome_Ratio_Norm_gene_Median.cct', 'HS_CPTAC_BRCA_2018_RNA_GENE.cct', 'README.md', 'data', 'module1_setup.ipynb']


In [15]:
# --- sanity check: confirm the three files are ready and look right ---
for f in ["cptac_brca_rna.tsv", "cptac_brca_protein.tsv", "cptac_brca_mutation.tsv"]:
    d = pd.read_csv(os.path.join(OUT_DIR, f), sep="\t", index_col=0)
    print(f"{f:28s} shape={d.shape}  missing={d.isna().mean().mean():.1%}  TP53 present: {'TP53' in d.index}")

shared = rna.columns.intersection(prot.columns)
print(f"\nRNA/protein shared tumor samples: {len(shared)}  (should be ~122)")
print(f"All three files are now in {OUT_DIR}/. You're done here.")

cptac_brca_rna.tsv           shape=(23121, 122)  missing=21.6%  TP53 present: True
cptac_brca_protein.tsv       shape=(12621, 122)  missing=19.2%  TP53 present: True
cptac_brca_mutation.tsv      shape=(9448, 1)  missing=0.0%  TP53 present: True

RNA/protein shared tumor samples: 122  (should be ~122)
All three files are now in the data/ folder. You're done here.


### Done ✓
The three files are in `data/part1_cptac_brca/`. Now open **`BIOT6900_Module2_Starter.ipynb`** and run `Kernel → Restart & Run All` — Part 1 will now use your real CPTAC data instead of the synthetic demo.

Watch for: a modest RNA–protein correlation (typically ~0.4–0.5), and familiar breast-cancer genes (TP53, ERBB2, GATA3, PIK3CA) near the top of the ranked list.